# COURSE: A deep understanding of deep learning
## SECTION: ANNs
### LECTURE: ANN for classifying qwerties
#### TEACHER: Mike X Cohen, sincxpress.com
##### COURSE URL: udemy.com/course/deeplearning_x/?couponCode=202401

In [1]:
# import libraries
import torch
import torch.nn as nn
import numpy as np

import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

In [ ]:
# create data

nPerClust = 100
blur = 1

A = [  1, 1 ]
B = [  5, 1 ]

# generate data
a = [ A[0]+np.random.randn(nPerClust)*blur , A[1]+np.random.randn(nPerClust)*blur ]
b = [ B[0]+np.random.randn(nPerClust)*blur , B[1]+np.random.randn(nPerClust)*blur ]

# true labels
labels_np = np.vstack((np.zeros((nPerClust,1)),np.ones((nPerClust,1))))

# concatanate into a matrix
data_np = np.hstack((a,b)).T

# convert to a pytorch tensor
data = torch.tensor(data_np).float()
labels = torch.tensor(labels_np).float()

# show the data
fig = plt.figure(figsize=(5,5))
plt.plot(data[np.where(labels==0)[0],0],data[np.where(labels==0)[0],1],'bs')
plt.plot(data[np.where(labels==1)[0],0],data[np.where(labels==1)[0],1],'ko')
plt.title('The qwerties!')
plt.xlabel('qwerty dimension 1')
plt.ylabel('qwerty dimension 2')
plt.show()

Generates the synthetic data for the classification task and visualizes it. Let's break it down:

----
----

    nPerClust = 100

This sets the number of data points in each of the two clusters to 100.

    blur = 1

This variable controls the spread or variance of the data points within each cluster. A higher value means more spread and overlap between clusters.

    A = [ 1, 1 ]
    B = [ 5, 1 ]

These lines define the mean (center) coordinates for the two clusters, labeled A and B.]

    a = [ A[0]+np.random.randn(nPerClust)*blur , A[1]+np.random.randn(nPerClust)*blur ]
    b = [ B[0]+np.random.randn(nPerClust)*blur , B[1]+np.random.randn(nPerClust)*blur ]

These lines generate the actual data points for each cluster.

They take the mean coordinates (A and B) and add random noise sampled from a standard normal distribution (np.random.randn(nPerClust)) scaled by the blur value.

This creates clusters of data points around the specified means.

    labels_np = np.vstack((np.zeros((nPerClust,1)),np.ones((nPerClust,1))))

 This creates the true labels for the data. It generates an array of zeros for the first cluster (A) and an array of ones for the second cluster (B), then stacks them vertically to create a single array of labels.

    data_np = np.hstack((a,b)).T
This combines the data points from clusters A and B into a single NumPy array called data_np. np.hstack((a,b)) stacks the two arrays horizontally, and .T transposes the result so that each row represents a data point with two features (dimensions).

    data = torch.tensor(data_np).float() and labels = torch.tensor(labels_np).float()

These lines convert the NumPy arrays data_np and labels_np into PyTorch tensors.

Converting to tensors is necessary because PyTorch models work with tensors. .float() ensures the data type is float, which is common for neural network inputs and labels.

The code then proceeds to plot the generated data using Matplotlib. It plots the data points from cluster A (labels 0) as blue squares ('bs') and data points from cluster B (labels 1) as black circles ('ko'). It also adds a title and axis labels to the plot.

In essence, this cell sets up the problem by creating two separable clusters of data points and their corresponding labels, and then it visualizes these points to show their distribution.



In [ ]:
# inspect types
print(type(data_np))
print(np.shape(data_np))
print(' ')

print(type(data))
print(np.shape(data))

In [ ]:
# build the model
ANNclassify = nn.Sequential(
    nn.Linear(2,1),   # input layer
    nn.ReLU(),        # activation unit
    nn.Linear(1,1),   # output unit
    nn.Sigmoid(),     # final activation unit (here for conceptual reasons; in practice, better to use BCEWithLogitsLoss)
      )

ANNclassify

In [ ]:
# other model features

learningRate = .01

# loss function
lossfun = nn.BCELoss()
# Note: You'll learn in the "Metaparameters" section that it's better to use BCEWithLogitsLoss, but this is OK for now.

# optimizer
optimizer = torch.optim.SGD(ANNclassify.parameters(),lr=learningRate)


# Defines the optimizer and loss function for the neural network.

----
----

    learningRate = .01
This sets the learning rate for the optimizer. The learning rate is a hyperparameter that controls how much the model's weights are adjusted during training.

A smaller learning rate means smaller adjustments, and a larger learning rate means larger adjustments.

    lossfun = nn.BCELoss()
This defines the loss function that will be used to measure the difference between the model's predictions and the true labels. nn.BCELoss() stands for Binary Cross-Entropy Loss, which is commonly used for binary classification problems like this one.

    optimizer = torch.optim.SGD(ANNclassify.parameters(),lr=learningRate)

 This defines the optimizer that will be used to update the model's weights during training. torch.optim.SGD() creates a Stochastic Gradient Descent (SGD) optimizer.

 It takes the model's parameters (ANNclassify.parameters()) and the learning rate (lr=learningRate) as arguments.

 The parameters() method in PyTorch, when called on a module (like your ANNclassify model), is a very important function. It returns an iterator over all the learnable parameters (weights and biases) of the module and its submodules.

In the context of your code, when you use optimizer = torch.optim.SGD(ANNclassify.parameters(),lr=learningRate), you are telling the SGD optimizer which parameters it should update during the training process. The optimizer needs to know which tensors in your model are the ones that need their values adjusted based on the calculated gradients to minimize the loss.

Think of it this way: your ANNclassify model is made up of layers (nn.Linear). Each of these layers has internal weights and biases that the model learns during training. The parameters() method gathers all these learnable weights and biases from across all the layers in your model and presents them to the optimizer.

So, ANNclassify.parameters() provides the optimizer with a list or collection of all the numbers (parameters) within your neural network that need to be tuned during the learning process to improve the model's performance.

What can I help you build?


 The optimizer's job is to minimize the loss function by adjusting the model's parameters based on the gradients calculated during backpropagation.

In [ ]:
# train the model
numepochs = 1000
losses = torch.zeros(numepochs)

for epochi in range(numepochs):

  # forward pass
  yHat = ANNclassify(data)

  # compute loss
  loss = lossfun(yHat,labels)
  losses[epochi] = loss

  # backprop
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

In [ ]:
# show the losses

plt.plot(losses.detach(),'o',markerfacecolor='w',linewidth=.1)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
# compute the predictions

# manually compute losses
# final forward pass
predictions = ANNclassify(data)

predlabels = predictions>.5

# find errors
misclassified = np.where(predlabels != labels)[0]

# total accuracy
totalacc = 100-100*len(misclassified)/(2*nPerClust)

print('Final accuracy: %g%%' %totalacc)


In [ ]:
# plot the labeled data
fig = plt.figure(figsize=(5,5))
plt.plot(data[misclassified,0] ,data[misclassified,1],'rx',markersize=12,markeredgewidth=3)
plt.plot(data[np.where(~predlabels)[0],0],data[np.where(~predlabels)[0],1],'bs')
plt.plot(data[np.where(predlabels)[0],0] ,data[np.where(predlabels)[0],1] ,'ko')

plt.legend(['Misclassified','blue','black'],bbox_to_anchor=(1,1))
plt.title(f'{totalacc}% correct')
plt.show()

# Additional explorations

In [ ]:
# 1) It is common in DL to train the model for a specified number of epochs. But you can also train until
#    the model reaches a certain accuracy criterion. Re-write the code so that the model continues training
#    until it reaches 90% accuracy.
#    What would happen if the model falls into a local minimum and never reaches 90% accuracy? Yikes! You can
#    force-quit a process in google-colab by clicking on the top-left 'play' button of a code cell.
#
# 2) It is intuitive that the model can reach 100% accuracy if the qwerties are more separable. Modify the
#    qwerty-generating code to get the model to have 100% classification accuracy.
#